# CSA0812 – Assignment 3, Q5
## List Comprehension: Cleaning Sensor Readings

**Student:** Darshan A  
**Register Number:** 1973260011  
**Marks:** 6 | **Level:** K3 – Apply

### Aim
Clean cold-storage temperature readings by removing sensor errors outside -50°C to 50°C, convert valid values to Fahrenheit, and identify alarm minutes where valid temperatures are outside 2°C to 8°C.

### Prediction
The loop, list-comprehension and map/filter approaches should produce identical results. All three require O(n) time. List comprehensions create a complete list in memory, while generator expressions produce values lazily and use much less additional memory.

## 1. Main implementation: loop, list comprehension, and map/filter

The three versions use the same validity rule: -50 <= temperature <= 50. Fahrenheit is calculated using (C × 9/5) + 32. Alarm minutes are zero-based positions in the original readings, found with enumerate().

In [1]:
readings = [
    -60, 4.0, 5.5, 9.0, 52, 7.2, 1.5, -51, 3.8, 10.0,
    6.0, 55, 2.0, 8.0, 8.5, -4.0, 51, 4.5, 6.8, 0.5
]

# Version 1: for loop with append()
valid_loop = []
fahrenheit_loop = []
for celsius in readings:
    if -50 <= celsius <= 50:
        valid_loop.append(celsius)
        fahrenheit_loop.append((celsius * 9 / 5) + 32)

# Version 2: list comprehensions with conditions
valid_comp = [c for c in readings if -50 <= c <= 50]
fahrenheit_comp = [(c * 9 / 5) + 32 for c in readings if -50 <= c <= 50]

# Version 3: filter() and map() with lambda
valid_map_filter = list(filter(lambda c: -50 <= c <= 50, readings))
fahrenheit_map_filter = list(
    map(lambda c: (c * 9 / 5) + 32, valid_map_filter)
)

# Alarm minutes: zero-based minute positions in the original readings.
# Sensor-error readings are excluded from the temperature alarm list.
alarm_minutes = [
    minute for minute, temp in enumerate(readings)
    if -50 <= temp <= 50 and not (2 <= temp <= 8)
]

print("Valid Celsius readings:", valid_comp)
print("Valid Fahrenheit readings:", [round(x, 2) for x in fahrenheit_comp])
print("Alarm minutes (0-based):", alarm_minutes)
print("Loop and comprehension match:", valid_loop == valid_comp and fahrenheit_loop == fahrenheit_comp)
print("Comprehension and map/filter match:", valid_comp == valid_map_filter and fahrenheit_comp == fahrenheit_map_filter)

assert valid_loop == valid_comp == valid_map_filter
assert fahrenheit_loop == fahrenheit_comp == fahrenheit_map_filter
print("All three versions give identical results.")

Valid Celsius readings: [4.0, 5.5, 9.0, 7.2, 1.5, 3.8, 10.0, 6.0, 2.0, 8.0, 8.5, -4.0, 4.5, 6.8, 0.5]
Valid Fahrenheit readings: [39.2, 41.9, 48.2, 44.96, 34.7, 38.84, 50.0, 42.8, 35.6, 46.4, 47.3, 24.8, 40.1, 44.24, 32.9]
Alarm minutes (0-based): [3, 6, 9, 14, 15, 19]
Loop and comprehension match: True
Comprehension and map/filter match: True
All three versions give identical results.


## 2. Design examples: dictionary and set comprehensions

A dictionary comprehension can map each original minute to its valid Celsius reading. A set comprehension can collect the distinct valid readings.

In [2]:
valid_by_minute = {
    minute: temp
    for minute, temp in enumerate(readings)
    if -50 <= temp <= 50
}

distinct_valid_temperatures = {
    temp for temp in readings if -50 <= temp <= 50
}

print("Valid readings by minute:", valid_by_minute)
print("Distinct valid temperatures:", sorted(distinct_valid_temperatures))

Valid readings by minute: {1: 4.0, 2: 5.5, 3: 9.0, 5: 7.2, 6: 1.5, 8: 3.8, 9: 10.0, 10: 6.0, 12: 2.0, 13: 8.0, 14: 8.5, 15: -4.0, 17: 4.5, 18: 6.8, 19: 0.5}
Distinct valid temperatures: [-4.0, 0.5, 1.5, 2.0, 3.8, 4.0, 4.5, 5.5, 6.0, 6.8, 7.2, 8.0, 8.5, 9.0, 10.0]


## 3. Experimental investigation: 10,000, 100,000 and 1,000,000 readings

Each method receives the same generated readings. The median of three runs is measured. Results are checked for equality before the timings are reported.

In [3]:
import random
import time
import statistics

random.seed(42)

def make_readings(size):
    # Includes valid values and sensor errors
    return [random.uniform(-60, 60) for _ in range(size)]

def clean_loop(data):
    result = []
    for value in data:
        if -50 <= value <= 50:
            result.append(value)
    return result

def clean_comprehension(data):
    return [value for value in data if -50 <= value <= 50]

def clean_map_filter(data):
    return list(filter(lambda value: -50 <= value <= 50, data))

def median_runtime(function, data, repeats=3):
    times = []
    result = None
    for _ in range(repeats):
        start = time.perf_counter()
        result = function(data)
        times.append(time.perf_counter() - start)
    return statistics.median(times), result

benchmark_results = []
for size in (10_000, 100_000, 1_000_000):
    data = make_readings(size)

    loop_time, loop_result = median_runtime(clean_loop, data)
    comp_time, comp_result = median_runtime(clean_comprehension, data)
    map_time, map_result = median_runtime(clean_map_filter, data)

    match = loop_result == comp_result == map_result
    assert match

    benchmark_results.append({
        "Readings": size,
        "Valid readings": len(comp_result),
        "Loop (s)": loop_time,
        "List comprehension (s)": comp_time,
        "map/filter (s)": map_time,
        "All results match": match
    })

print("Benchmark results (median of 3 runs):")
for result in benchmark_results:
    print(result)

Benchmark results (median of 3 runs):
{'Readings': 10000, 'Valid readings': 8375, 'Loop (s)': 0.0009400879999930112, 'List comprehension (s)': 0.0008820460000151797, 'map/filter (s)': 0.0011151400000244394, 'All results match': True}
{'Readings': 100000, 'Valid readings': 83167, 'Loop (s)': 0.009274593000014875, 'List comprehension (s)': 0.00864648699999293, 'map/filter (s)': 0.011390021000011075, 'All results match': True}
{'Readings': 1000000, 'Valid readings': 833552, 'Loop (s)': 0.09545239700000252, 'List comprehension (s)': 0.09140591299998846, 'map/filter (s)': 0.2558836050000082, 'All results match': True}


## 4. Memory comparison: list comprehension versus generator expression

sys.getsizeof() measures the size of the container object itself. The list stores all generated values, while the generator stores its iteration state and creates values as needed. The generator is consumed once in the sum example.

In [4]:
import sys

sample = [float(i % 101) for i in range(100_000)]

list_result = [c for c in sample if -50 <= c <= 50]
generator_result = (c for c in sample if -50 <= c <= 50)

print("List object size:", sys.getsizeof(list_result), "bytes")
print("Generator object size:", sys.getsizeof(generator_result), "bytes")
print("List result count:", len(list_result))
print("Generator sum:", sum(generator_result))

# Generator-expression alternative: process valid values without storing a list.
valid_temperature_sum = sum(c for c in sample if -50 <= c <= 50)
print("Sum using generator expression:", valid_temperature_sum)

List object size: 444376 bytes
Generator object size: 208 bytes
List result count: 50500
Generator sum: 1262295.0
Sum using generator expression: 1262295.0


## 5. Edge-case tests: all-valid and all-invalid readings

In [5]:
all_valid = [2.0, 3.5, 5.0, 7.5, 8.0, -10.0, 0.0]
all_invalid = [51.0, -51.0, 60.0, -80.0]

for label, test_data in [("All valid", all_valid), ("All invalid", all_invalid)]:
    loop_result = clean_loop(test_data)
    comp_result = clean_comprehension(test_data)
    map_result = clean_map_filter(test_data)

    print(label)
    print("Loop:", loop_result)
    print("Comprehension:", comp_result)
    print("Map/filter:", map_result)
    print("All methods match:", loop_result == comp_result == map_result)
    print()

    assert loop_result == comp_result == map_result

assert clean_comprehension(all_valid) == all_valid
assert clean_comprehension(all_invalid) == []
print("All-valid and all-invalid tests passed.")

All valid
Loop: [2.0, 3.5, 5.0, 7.5, 8.0, -10.0, 0.0]
Comprehension: [2.0, 3.5, 5.0, 7.5, 8.0, -10.0, 0.0]
Map/filter: [2.0, 3.5, 5.0, 7.5, 8.0, -10.0, 0.0]
All methods match: True

All invalid
Loop: []
Comprehension: []
Map/filter: []
All methods match: True

All-valid and all-invalid tests passed.


## 6. Design and analysis

### General form of a conditional list comprehension
`[expression for item in iterable if condition]`

The expression is evaluated and included only when the condition is true.

### When can a comprehension be harder to read than a loop?
A comprehension becomes difficult to read when it contains several conditions, nested loops, complicated transformations, or side effects. A normal loop is clearer when multiple steps, error handling or debugging are needed.

### Dictionary and set comprehensions
- Dictionary: `{minute: temp for minute, temp in enumerate(readings) if -50 <= temp <= 50}`
- Set: `{temp for temp in readings if -50 <= temp <= 50}`

### List comprehension versus generator expression
A list comprehension builds and stores the complete result immediately. A generator expression produces values lazily when requested. Both take O(n) time to process n readings, but a list uses O(n) result memory while a generator uses O(1) additional iteration memory, excluding the input data.

### Complexity
- Cleaning with a loop: O(n) time and O(n) output space.
- List comprehension: O(n) time and O(n) output space.
- map/filter converted to a list: O(n) time and O(n) output space.
- Generator expression: O(n) time when fully consumed and O(1) additional iteration space.

### Conclusion
A list comprehension is concise and suitable for creating a reusable cleaned list. It is not always the best choice: a generator is preferable when processing a large stream once and memory use matters, while a normal loop can be easier to understand for multi-step logic. Use the measured timings and memory values printed by this notebook to support the final comparison.